# dFF Skewness Distribution by Layer (Layer 2/3 vs Layer 5)

This notebook loads the calcium imaging `dFF` traces of all cell ROIs across motor recording sessions in projects `ccyp_l5_3d_vision` and `colasa_3d-vision_revisions` (excluding `PZAG22.1b_S20260220`). It then:
1. Filters ROIs to keep only those identified as actual cells (`iscell == True`).
2. Computes the skewness of the dFF trace over time for each cell ROI.
3. Categorizes sessions into **Layer 2/3** (nominal depth <= 350 µm) and **Layer 5** (nominal depth > 350 µm).
4. Plots and compares the distribution of skewness between layers.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import flexiznam as flz
from cottage_analysis.summary_analysis.get_session_list import get_motor_session_list
from cottage_analysis.io_module.suite2p import load_is_cell

# Set style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording, recording was wrongly started"
}

In [ ]:
# 1. Fetch sessions for the projects
project_ccyp = "ccyp_l5_3d_vision"
project_colasa = "colasa_3d-vision_revisions"

print("Fetching sessions from flexilims...")
flz_sess_ccyp = flz.get_flexilims_session(project_id=project_ccyp)
sessions_ccyp = get_motor_session_list(flz_sess_ccyp)

flz_sess_colasa = flz.get_flexilims_session(project_id=project_colasa)
sessions_colasa = get_motor_session_list(flz_sess_colasa)

# Combine and exclude
session_info = []
for s in sessions_ccyp:
    if s not in SESSIONS_TO_EXCLUDE:
        session_info.append((s, project_ccyp, flz_sess_ccyp))
for s in sessions_colasa:
    if s not in SESSIONS_TO_EXCLUDE:
        session_info.append((s, project_colasa, flz_sess_colasa))

# Remove duplicates
session_info = list(dict.fromkeys(session_info))

print(f"Total sessions to process: {len(session_info)}")
for s, proj, _ in session_info:
    print(f" - {s} (Project: {proj})")

In [ ]:
def load_session_dff_and_iscell(session_name, project_id, flexilims_session):
    # Get the nominal depth of the session
    project_sessions = flz.get_entities("session", flexilims_session=flexilims_session)
    if session_name in project_sessions.index:
        nominal_depth = project_sessions.loc[session_name, "nominal_depth"]
        if isinstance(nominal_depth, (list, np.ndarray, pd.Series)):
            nominal_depth = np.mean(nominal_depth)
    else:
        print(f"Session {session_name} not found in session entities.")
        return None
    
    # Get the suite2p_rois dataset
    suite2p_datasets = flz.get_datasets(
        origin_name=session_name,
        dataset_type="suite2p_rois",
        project_id=project_id,
        flexilims_session=flexilims_session,
        return_dataseries=False,
        filter_datasets={"anatomical_only": 3},
    )
    if not suite2p_datasets:
        suite2p_datasets = flz.get_datasets(
            origin_name=session_name,
            dataset_type="suite2p_rois",
            project_id=project_id,
            flexilims_session=flexilims_session,
            return_dataseries=False,
        )
    if not suite2p_datasets:
        print(f"No suite2p_rois dataset found for {session_name}")
        return None
    
    suite2p_ds = suite2p_datasets[0]
    suite2p_path = Path(suite2p_ds.path_full)
    
    # Load iscell
    try:
        iscell = load_is_cell(suite2p_path)
    except Exception as e:
        print(f"Error loading iscell for {session_name}: {e}")
        return None
        
    # Get number of planes
    nplanes = int(suite2p_ds.extra_attributes.get("nplanes", 1))
    
    # Determine the file name for dff (dff_ast.npy or dff.npy)
    dff_fname = (
        "dff_ast.npy" if suite2p_ds.extra_attributes.get("ast_neuropil", False) else "dff.npy"
    )
    
    # Load and stack plane-by-plane dff
    dffs = []
    for iplane in range(nplanes):
        dff_file = suite2p_path / f"plane{iplane}" / dff_fname
        if not dff_file.exists():
            alt_fname = "dff.npy" if dff_fname == "dff_ast.npy" else "dff_ast.npy"
            dff_file_alt = suite2p_path / f"plane{iplane}" / alt_fname
            if dff_file_alt.exists():
                dff_file = dff_file_alt
            else:
                print(f"No dff found for plane {iplane} in {session_name}. Skipping plane.")
                continue
        dffs.append(np.load(dff_file))
        
    if not dffs:
        print(f"No dff files loaded for {session_name}")
        return None
        
    # Stack along the ROI axis. Shape: (total_rois, n_frames)
    dff_stacked = np.vstack(dffs)
    
    # Check shape match with iscell
    if len(iscell) != dff_stacked.shape[0]:
        print(f"Warning: iscell length ({len(iscell)}) does not match dff ROI count ({dff_stacked.shape[0]}) for {session_name}")
        min_len = min(len(iscell), dff_stacked.shape[0])
        iscell = iscell[:min_len]
        dff_stacked = dff_stacked[:min_len, :]
        
    return {
        "dff": dff_stacked,
        "iscell": iscell,
        "nominal_depth": nominal_depth
    }

In [ ]:
# 2. Process all sessions and compute skewness
all_skewness = []

for session_name, project_id, flz_sess in tqdm(session_info, desc="Processing sessions"):
    print(f"Loading {session_name}...")
    res = load_session_dff_and_iscell(session_name, project_id, flz_sess)
    if res is None:
        continue
    
    dff = res["dff"]
    iscell = res["iscell"]
    nominal_depth = res["nominal_depth"]
    
    # Filter for cells
    dff_cells = dff[iscell]
    
    # Compute skewness, mean, 90th percentile, and max along the frame/time axis (axis=1)
    skew_vals = stats.skew(dff_cells, axis=1, nan_policy='omit')
    mean_vals = np.nanmean(dff_cells, axis=1)
    max_vals = np.nanmax(dff_cells, axis=1)
    std_diff = np.nanstd(np.diff(dff_cells, axis=1), axis=1)
    std_vals = np.nanstd(dff_cells, axis=1)
    
    for roi_idx, skew_val in enumerate(skew_vals):
        all_skewness.append({
            "session": session_name,
            "project": project_id,
            "roi": roi_idx,
            "skewness": skew_val,
            "mean_dff": mean_vals[roi_idx],
            "std_diff": std_diff[roi_idx],
            "std_vals": std_vals[roi_idx],
            "max_dff": max_vals[roi_idx],
            "nominal_depth": nominal_depth
        })

skew_df = pd.DataFrame(all_skewness)
skew_df = skew_df.dropna(subset=['skewness'])

# 3. Categorize by nominal depth
cut_off = 350
skew_df["layer"] = np.where(skew_df["nominal_depth"] <= cut_off, "Layer 2/3", "Layer 5")

print(f"\nProcessing complete! Found {len(skew_df)} cell ROIs across {skew_df['session'].nunique()} sessions.")

In [ ]:
# 4. Plot the skewness distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: KDE/Histogram overlay
sns.histplot(
    data=skew_df,
    x="skewness",
    hue="layer",
    kde=False,
    common_norm=False,
    stat='density',
    element="step",
    alpha=0.3,
    linewidth=2,
    palette={"Layer 2/3": "C0", "Layer 5": "C1"},
    ax=axes[0]
)
axes[0].set_title("dFF Skewness Density by Layer")
axes[0].set_xlabel("Skewness per ROI")
axes[0].set_ylabel("Density")
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)
axes[0].semilogy()
# Plot 2: Violin plot showing quartiles
sns.histplot(
    data=skew_df,
    x="skewness",
    hue="layer",
    kde=False,
    cumulative=True,
    common_norm=False,
    stat='density',
    element="step",
    alpha=0.3,
    linewidth=2,
    palette={"Layer 2/3": "C0", "Layer 5": "C1"},
    ax=axes[1]
)
plt.xlim(-1, 30)
plt.tight_layout()
plt.show()

In [ ]:
l5 = skew_df[skew_df.layer=="Layer 5"]
l23= skew_df[skew_df.layer=="Layer 2/3"]
plt.scatter(l5.std_diff, l5.std_diff / l5.std_vals,alpha=0.2, label='Layer5', s=2)
plt.scatter(l23.std_diff, l23.std_diff / l23.std_vals,  alpha=0.2, label='Layer2/3', s=2)

plt.legend(loc='lower right')
plt.loglog()

In [ ]:
l5.std_vals/l5.std_diff

In [ ]:
plt.hist((l5.std_vals/l5.std_diff).values, density=True, bins=np.arange(0.5,3,0.01), cumulative=True, alpha=0.95, histtype='step')
_ = plt.hist( (l23.std_vals/ l23.std_diff).values, density=True, bins=np.arange(0.5,3,0.01), cumulative=True, alpha=0.95, histtype='step')

In [ ]:
# 5. Plot distributions of Mean, 90% percentile, and Max dFF per ROI by Layer
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

metrics = [
    ("std_diff", "Mean dFF", (-0.5, 10)),
    ("std_vals", "90th Percentile dFF", (-0.5, 10)),
    ("max_dff", "Max dFF", (-0.5, 15.0))
]

palette = {"Layer 2/3": "C0", "Layer 5": "C1"}

for row, (col_name, label, xlim) in enumerate(metrics):
    # Histplot/KDE
    sns.histplot(
        data=skew_df,
        x=col_name,
        hue="layer",
        cumulative=True,
        kde=False,
        common_norm=False,
        element="step",
        alpha=0.3,
        linewidth=2,
        palette=palette,
        ax=axes[row, 0]
    )
    axes[row, 0].set_title(f"{label} Density by Layer")
    axes[row, 0].set_xlabel(f"{label} per ROI")
    axes[row, 0].set_ylabel("Density")
    axes[row, 0].spines['top'].set_visible(False)
    axes[row, 0].spines['right'].set_visible(False)

    if xlim:
        axes[row, 0].set_xlim(xlim)

    # Violinplot
    sns.violinplot(
        data=skew_df,
        x="layer",
        y=col_name,
        palette=palette,
        inner="quartile",
        ax=axes[row, 1]
    )
    axes[row, 1].set_title(f"{label} Comparison")
    axes[row, 1].set_xlabel("Layer")
    axes[row, 1].set_ylabel(f"{label} per ROI")
    axes[row, 1].spines['top'].set_visible(False)
    axes[row, 1].spines['right'].set_visible(False)
    if xlim:
        axes[row, 1].set_ylim(xlim)

plt.tight_layout()
plt.show()


In [ ]:
sns.histplot(data=skew_df, x="nominal_depth", hue='layer')